## Notebook 5 of 5 — Before/After Comparison, Figures & Export

Produces the final comparison figures (OLS scatter facets by ecosystem, fair site-matched before/after calibration comparison, performance vs. elevation, AET error as a fraction of precipitation), a PET diagnostics tail, and exports the final comprehensive metrics table to `Data/open_et/et_comparison_metrics.csv`.

*Part of a 5-notebook pipeline (run in order; each caches its outputs to `Data/` so later notebooks can be re-run without repeating expensive GEE/download steps): `01_select_parks_towers` -> `02_load_flux_openet_gridmet` -> `03_run_wbm` -> `04_validate_and_calibrate` -> `05_figures_and_export`.*

<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2025/blob/main/lectures/lecture4-ET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CIVE 523 Final Project – Kristen Cognac, February 27, 2026

Objective: The National Park Service (NPS) Water Balance Model (WBM) employs a set of parameterizations and functions to estimate a daily water balance for components rain, snow, snowmelt, soil water storage, evapotranspiration, and lumped runoff (surface water and groundwater). Actual evapotranspiration (ETa) is estimated using a simple “bucket-type” approach wherein potential evapotranspiration (Oudin, 2005) is maximized to the extent of available soil moisture storage. The NPS WBM has been recently applied across parks to assess past and future changes in water availability (e.g., Thoma, 2019; Thoma, 2020). However, detailed validation of model components has yet to be conducted. 

OpenET, with daily estimates of ETa from six remote sensing models, provides a robust, spatially continuous dataset for validating NPS WBM ETa. This project will evaluate NPS WBM ETa accuracy by comparing it to OpenET monthly and annual timeseries using common statistical metrics. Given that OpenET has errors too, OpenET accuracy will also be assessed using ground-truth measurements from flux towers. The implications of ETa inaccuracy on water availability assessments will be considered by comparing error magnitude to the total water balance at each location. This project will rely heavily on previous data and analysis compiled by Volk et al. (2024), which assessed the accuracy of OpenET across CONUS. In particular, they provide corrected timeseries of in situ ETa (and OpenET) through Zenodo repositories that are useful for estimating OpenET accuracy (Volk et al., 2023a; 2023b).

Methods: Six US National Park sites within 20 km of eddy covariance towers (AmeriFlux, USGS NWSC) are selected for comparison. Towers and park elevations range from 0 to 4,350m, enabling assessment across an elevation gradient. Daily ETa (1999-2024) will be calculated for each park centroid (4km GridMET) using the NPS WBM. The single grid cell minimizes errors from variable park sizes and limits data processing. (Note: Generating daily NPS WBM ETa requires downloading GridMET timeseries, extracting parameters (soil water capacity, slope, and elevation), and running WBM functions). The aerial analysis extent is 80 km2 distributed across the five parks. OpenET (30m grid) will be spatially averaged for each 4km2 cell to generate daily, monthly, and annual Eta. NPS WBM ETa accuracy will be assessed using: linear regression slope (forced through the origin), mean bias error (MBE), mean absolute error (MAE), root-mean-square error (RMSE), and coefficient of determination (r2). Flux tower data (Volk et al., 2023a) will be resampled to daily, monthly, and annual values and compared to the corresponding OpenET ETa pixel (Volk et al., 2023b). OpenET accuracy will be similarly assessed using linear regression, MBE, MAE, RMSE, and r2). Relative error metrics (e.g., MAE/Precipitation) will be evaluated to understand impacts on water availability assessments.

References:

Oudin, L., Hervieu, F., Michel, C., Perrin, C., Andréassian, V., Anctil, F., & Loumagne, C. (2005). Which potential evapotranspiration input for a lumped rainfall–runoff model?: Part 2—Towards a simple and efficient potential evapotranspiration model for rainfall–runoff modelling. Journal of hydrology, 303(1-4), 290-306.

Thoma, D. P., Munson, S. M., & Witwicki, D. L. (2019). Landscape pivot points and responses to water balance in national parks of the southwest US. Journal of Applied Ecology, 56(1), 157-167.

Thoma, D. P., Tercek, M. T., Schweiger, E. W., Munson, S. M., Gross, J. E., & Olliff, S. T. (2020). Water balance as an indicator of natural resource condition: Case studies from Great Sand Dunes National Park and Preserve. Global Ecology and Conservation, 24, e01300.

John M. Volk, Justin L. Huntington, Forrest Melton, Blake Minor, Tianxin Wang, Saseendran S. Anapalli, Raymond G. Anderson, Steven R. Evett, Andrew N. French, Richard Jasoni, Nicolas Bambach, William P. Kustas, Joseph G. Alfieri, John Prueger, Lawrence Hipps, Lynn McKee, Sebastian J. Castro Bustamante, Maria del Mar Alsina, Andrew McElrone, … Martha Anderson. (2023a). Post-processed data and graphical tools for a CONUS-wide eddy flux evapotranspiration dataset (1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.7636781

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., Kilic, A., Ruhoff, A., Senay, G. B., Minor, B., Morton, C., Ott, T., Johnson, L., Andrade, B. C. D., Carrara, W., Doherty, C. T., Dunkerly, C., Friedrichs, M., Guzman, A., … Yang, Y. (2023b). OpenET model data for assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10119477

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., ... & Yang, Y. (2024). Assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications. Nature Water, 2(2), 193-205.





# Setup Workspace

Import necessary libraries.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMPORTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Make the repo root (parent of notebooks/) importable ─────────────────────
import sys
from pathlib import Path as _Path
_REPO_ROOT = _Path.cwd().parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# ── Standard library ──────────────────────────────────────────────────────────
import importlib
import os
import time
import tempfile
import warnings
import zipfile
from datetime import date
from pathlib import Path

# ── Scientific computing ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr

# ── Geospatial ────────────────────────────────────────────────────────────────
import geopandas as gpd
import rasterio
import shapely
from rasterio.transform import rowcol
from shapely.geometry import Point
from adjustText import adjust_text

# ── Google Earth Engine ───────────────────────────────────────────────────────
import ee
import geemap

# ── Visualization ─────────────────────────────────────────────────────────────
import branca.colormap as cm
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# ── Statistics & optimization ─────────────────────────────────────────────────
import requests
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import pdist, squareform

# ── Utilities ─────────────────────────────────────────────────────────────────
from adjustText import adjust_text
from tqdm import tqdm

# ── NPS WBM (local package, repo_root/wbm/) ───────────────────────────────────
import wbm
from wbm import (
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run
)


Authenticate Google Earth Engine

In [ ]:
#if not ee.data._credentials:
ee.Authenticate()
ee.Initialize(project='modis-475315')

Define helper functions

In [ ]:
# functions needed for this workflow

# this function is used to add a google earth engine layer to an existing folium map,
# for visualization purposes. Folium is a python package that can put rasters/shapefiles on a basemap
# the function below is run using an existing folium map. If the folium map defines is my_map, then
# my_map.add_ee_layer(ee_object,name)
# where ee_object is the object defined in google earth engine, and name is the label in folium
def add_ee_layer(self, ee_object, name):
    try:
        # display ee.Image()
        if isinstance(ee_object, ee.image.Image):
            range = ee.Image(ee_object).reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
            vals = range.getInfo()
            min=list(vals.items())[0][1]
            max=list(vals.items())[1][1]
            vis = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}

            map_id_dict = ee.Image(ee_object).getMapId(vis)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
            colormap = cm.LinearColormap(vmin=min,vmax=max,colors=['blue', 'white','red']).to_step(n=10)
            colormap.caption=name
            self.add_child(colormap)
        # display ee.ImageCollection()
        elif isinstance(ee_object, ee.imagecollection.ImageCollection):
            ee_object_new = ee_object.mosaic()
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
        # display ee.Geometry()
        elif isinstance(ee_object, ee.geometry.Geometry):
            folium.GeoJson(
            data = ee_object.getInfo(),
            name = name,
            overlay = True,
            control = True
        ).add_to(self)
        # display ee.FeatureCollection()
        elif isinstance(ee_object, ee.featurecollection.FeatureCollection):
            ee_object_new = ee.Image().paint(ee_object, 0, 2)
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
        ).add_to(self)

    except Exception as e:
        print("Could not display {}".format(name))
        print(e)


# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  min=list(vals.items())[0][1]
  max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
 # range = img.reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
 # vals = range.getInfo()
 # min=list(vals.items())[0][1]
 # max=list(vals.items())[1][1]
 # visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(img)

# load prism data
def get_prism_image(date1,date2,geometry):

  prism = ee.ImageCollection('OREGONSTATE/PRISM/AN81m')
  prism_img = prism.filterDate(date1,date2).select('ppt').mean().clip(geometry)
  return(prism_img) # returns prism average monthly precipitation, in mm

# load landsat 8 data
def get_l8_image(date1,date2,geometry):

  l8 = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT')
  l8_img = l8.filterDate(date1,date2).mean().clip(geometry)
  return(l8_img)

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

# to create an elevation raster from the USGS NED in google earth engine from a user-defined geometry
def get_elev(geometry):

  elev = ee.Image('USGS/NED').clip(geometry)
  return(elev)

# to create an elevation raster from the SRTM in google earth engine from a user-defined geometry
def get_srtm(geometry):

  elev = ee.Image('USGS/SRTMGL1_003').clip(geometry)
  return(elev)

# to create a temporally averaged precipitation raster from GPM from a user-defined geometry
def get_gpm_image(date1,date2,geometry):

  gpm = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07')
  gpm_img = gpm.filterDate(date1,date2).select('precipitation').mean().multiply(24*365/12).clip(geometry) # convert from mm/hour to mm/month
  return(gpm_img) # returns gpm average monthly precipitation in mm

# to create a temporally averaged actual ET raster from the openET ensemble from a user-defined geometry
def get_openET_image(date1,date2,geometry):

  openET = ee.ImageCollection('OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0')
  openET_img = openET.filterDate(date1,date2).select('et_ensemble_mad').mean().clip(geometry)
  return(openET_img)

# to create a temporally averaged reference ET raster from the openET ensemble from a user-defined geometry
def get_RET(date1,date2,geometry):

  ETR = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
  ETR_image = ETR.filterDate(date1,date2).select('etr').mean().multiply(365/12).clip(geometry) # convert from mm/day to mm/month
  return(ETR_image)

# load sentinel 2 data
def get_s2_image(date1,date2,geometry):

    s2 = ee.ImageCollection('COPERNICUS/S2')
    s2_img = s2.filterDate(date1,date2).filterBounds(geometry).first().clip(geometry)
    return(s2_img)

# Add EE drawing method to folium (not a function)
folium.Map.add_ee_layer = add_ee_layer

def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=1000,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):
  """Creates a region reduction function.

  Creates a region reduction function intended to be used as the input function
  to ee.ImageCollection.map() for reducing pixels intersecting a provided region
  to a statistic for each image in a collection. See ee.Image.reduceRegion()
  documentation for more details.

  Args:
    geometry:
      An ee.Geometry that defines the region over which to reduce data.
    reducer:
      Optional; An ee.Reducer that defines the reduction method.
    scale:
      Optional; A number that defines the nominal scale in meters of the
      projection to work in.
    crs:
      Optional; An ee.Projection or EPSG string ('EPSG:5070') that defines
      the projection to work in.
    bestEffort:
      Optional; A Boolean indicator for whether to use a larger scale if the
      geometry contains too many pixels at the given scale for the operation
      to succeed.
    maxPixels:
      Optional; A number specifying the maximum number of pixels to reduce.
    tileScale:
      Optional; A number representing the scaling factor used to reduce
      aggregation tile size; using a larger tileScale (e.g. 2 or 4) may enable
      computations that run out of memory with the default.

  Returns:
    A function that accepts an ee.Image and reduces it by region, according to
    the provided arguments.
  """

  def reduce_region_function(img):
    """Applies the ee.Image.reduceRegion() method.

    Args:
      img:
        An ee.Image to reduce to a statistic by region.

    Returns:
      An ee.Feature that contains properties representing the image region
      reduction results per band and the image timestamp formatted as
      milliseconds from Unix epoch (included to enable time series plotting).
    """

    stat = img.reduceRegion(
        reducer=reducer,
        geometry=geometry,
        scale=scale,
        crs=crs,
        bestEffort=bestEffort,
        maxPixels=maxPixels,
        tileScale=tileScale)

    return ee.Feature(geometry, stat).set({'millis': img.date().millis()})
  return reduce_region_function

# Define a function to transfer feature properties to a dictionary.
def fc_to_dict(fc):
  prop_names = fc.first().propertyNames()
  prop_lists = fc.reduceColumns(
      reducer=ee.Reducer.toList().repeat(prop_names.size()),
      selectors=prop_names).get('list')

  return ee.Dictionary.fromLists(prop_names, prop_lists)

# generate data frame from image collection
def gee_zonal_mean_img_coll(imageCollection,geometry,scale=1000):
    reduce_iC = create_reduce_region_function(geometry = geometry, scale=scale)
    stat_fc = ee.FeatureCollection(imageCollection.map(reduce_iC)).filter(ee.Filter.notNull(imageCollection.first().bandNames()))
    fc_dict = fc_to_dict(stat_fc).getInfo()

    df = pd.DataFrame(fc_dict)
    df['date'] = pd.to_datetime(df['millis'],unit='ms')
    return(df)

def gee_zonal_mean(date1,date2,geometry,collection_name,band_name,scale=1000):
     imcol = ee.ImageCollection(collection_name).select(band_name).filterDate(date1,date2)
     df = gee_zonal_mean_img_coll(imcol,geometry,scale=scale)
     return(df)

# Convert shapefile to EE object
def shapefile_to_ee(filepath):
    # 1. Read the shapefile
    gdf = gpd.read_file(filepath)
    
    # 2. Ensure it is in WGS84 (required by Earth Engine)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    
    # 3. Convert the geometry to a GeoJSON-like mapping
    # This handles Points, Polygons, and MultiPolygons
    geojson = gdf.__geo_interface__
    
    # 4. Create an ee.FeatureCollection from the GeoJSON
    # You can then get the geometry from the collection
    ee_object = ee.FeatureCollection(geojson)
    
    return ee_object.geometry()


# read in shapefile of National Parks
def get_park_system(save: bool = False, path: str | None = None) -> gpd.GeoDataFrame:
    """Import national park boundary shapefile.

    Downloads NPS park boundary shapefiles via the Data Store REST API.

    Reference:
        https://irmaservices.nps.gov/datastore/v6/documentation
        https://irma.nps.gov/DataStore/Reference/Profile/2224545?lnv=True

    Last updated: September 2025

    Args:
        save: If True, write the result to disk as a GeoPackage.
        path: Directory to save the file. Required when save=True.

    Returns:
        GeoDataFrame of all NPS park boundaries (EPSG:4326).
    """
    download_link = "https://irma.nps.gov/DataStore/DownloadFile/733895"

    # ── Download zip to a temp file ────────────────────────────────────────
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        zip_path = tmp_dir / "boundary.zip"
        extract_dir = tmp_dir / "extracted"

        response = requests.get(download_link, stream=True, timeout=120)
        response.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        # ── Unzip and read shapefile ───────────────────────────────────────
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        shp_path = extract_dir / "nps_boundary.shp"
        parks = gpd.read_file(shp_path)

        # ── Fix any invalid geometries (mirrors st_make_valid) ─────────────
        parks["geometry"] = parks["geometry"].make_valid()

        # ── Optionally save as GeoPackage ──────────────────────────────────
        if save and path is not None:
            out_path = (
                Path(path)
                / f"nps_park_boundaries_{date.today()}.gpkg"
            )
            parks.to_file(out_path, driver="GPKG")

    return parks


setup for NPS WBM

In [ ]:
# ── Import the WBM model from the `wbm` package (repo_root/wbm/) ────────────
# All functions are also available via the `wbm` namespace, e.g. wbm.nps_wbm(...)
import importlib
import wbm

# Reload in case you edited the package mid-session without restarting the kernel
importlib.reload(wbm)

from wbm import (
    # ── Raster utilities ──────────────────────────────────────────────────
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run

    # ── Core model ───────────────────────────────────────────────────────
    nps_wbm,                # single-point daily water balance driver
    run_nps_wbm_points,     # multi-point / multi-GCM wrapper

    # ── Component functions (available if needed for custom workflows) ────
    get_freeze,             # rain/snow partitioning factor
    get_rain, get_snow,     # rainfall and snowfall
    get_melt,               # Hock degree-day snowmelt
    get_snowpack,           # snowpack accumulation
    get_ablation,           # snow sublimation / vapor loss
    get_soil,               # soil water content
    get_d_soil,             # daily change in SWC
    get_aet,                # actual evapotranspiration
    get_storage,            # linear storage reservoir (CSU addition)
    get_oudin_pet,          # Oudin PET with topographic heat-load
    get_hamon_pet,          # Hamon PET
    get_penman_monteith_pet,# FAO-56 Penman-Monteith PET
    get_daylength,          # astronomical daylength (hours)
    get_gdd,                # growing degree days
    get_deficit,            # climatic water deficit (PET − AET)

    # ── Low-level helpers (rarely called directly) ────────────────────────
    get_svp, actual_vp, atm_press, psyc_constant,
    vapor_curve, clear_sky_rad, outgoing_rad,
)

print(f"WBM functions loaded from: {_REPO_ROOT / 'wbm'}")


Load raster datasets used for the NPS WBM

In [ ]:

# Reads the three required GeoTIFFs from `Data/wbm_rasters/`, derives slope
# and aspect from the DEM (Horn's 8-neighbour method, matching
# `terra::terrain(neighbors = 8)`), and caches everything in memory.
#
# **Only needs to run once per session.**  After this, `extract_point_params()`
# works without any path arguments.
#
# | File | Content | Units |
# |---|---|---|
# | `elevation_cropped.tif` | DEM | metres |
# | `water_storage.tif` | Soil water storage capacity | cm (auto-converted → mm) |
# | `merged_jennings2.tif` | Jennings temperature climatology | °C |

# %%
# ── Path to raster directory ──────────────────────────────────────────────────
# Adjust if your rasters live elsewhere.
RASTER_DIR = "../Data/wbm_rasters"

# ── Target CRS ────────────────────────────────────────────────────────────────
# Set to None to keep the native CRS of the rasters (EPSG:4326 for NPS data).
# Set to an EPSG string (e.g. "EPSG:26913") to reproject all rasters before
# sampling — useful when your point coordinates are in a projected CRS.
TARGET_CRS = "EPSG:4326"

try:
    load_wbm_rasters(raster_dir=RASTER_DIR, target_crs=TARGET_CRS)
except ImportError as e:
    print(f"⚠️  Rasters not loaded: {e}")
    print("   Install rasterio and re-run this cell before calling extract_point_params().")
except FileNotFoundError as e:
    print(f"⚠️  Raster file not found:\n   {e}")
    print(f"   Check that RASTER_DIR = '{RASTER_DIR}' is correct.")

Define global model settings for NPS WBM

In [ ]:
# Set default WBM run parameters here.  These are passed through to
# `nps_wbm()` / `run_nps_wbm_points()` / `run_pipeline()` in later cells.
# Override any of them at call-time as needed.


# ── PET method ────────────────────────────────────────────────────────────────
# One of: "Oudin"  (default, temperature-based, topographic heat-load adjusted)
#         "Hamon"  (temperature + daylength)
#         "Penman-Monteith"  (requires tmax, tmin, and ideally RH / wind data)
PET_METHOD = "Oudin"

# ── Snowmelt ──────────────────────────────────────────────────────────────────
# Hock (2003) degree-day melt factor (mm °C⁻¹ day⁻¹).
# Hock reports ~2.5 for Gooseberry Creek, UT; NPS default is 4.
HOCK_COEF = 4.0

# ── Initial conditions ────────────────────────────────────────────────────────
SNOWPACK_INIT = 0.0   # mm SWE
SOIL_INIT     = 0.0   # mm

# ── CSU additions ─────────────────────────────────────────────────────────────
DIRECT_FRAC  = 0.0    # fraction of rainfall routed directly to runoff (0–1)
RETURN_RATE  = 1.0    # fraction of storage reservoir released per day (0–1]
PET_MULT     = 1    # multiplicative PET bias correction
SOIL_MULT    = 1    # multiplicative adjustment to SWC_Max

# ── Misc ──────────────────────────────────────────────────────────────────────
SHADE_COEFF  = 1.0    # canopy shading coefficient for Oudin PET (0–1)
T_BASE       = 0.0    # base temperature for growing degree days (°C)
TO_INCHES    = True   # True → output fluxes in inches; False → mm

print("✓ Model settings configured:")
print(f"  PET method   : {PET_METHOD}")
print(f"  Hock coef    : {HOCK_COEF} mm °C⁻¹ day⁻¹")
print(f"  Direct frac  : {DIRECT_FRAC}")
print(f"  Return rate  : {RETURN_RATE}")
print(f"  PET mult     : {PET_MULT}   |  Soil mult: {SOIL_MULT}")
print(f"  Output units : {'inches' if TO_INCHES else 'mm'}")


In [ ]:
# Verify setup (quick sanity check)
#
# Runs the model for one synthetic year at a single point to confirm the full
# stack (imports → raster cache → model) is working before you connect real
# climate data.

# %%
rng = np.random.default_rng(0)
_n  = 365
_dates = pd.date_range("2000-01-01", periods=_n)
_doy   = _dates.dayofyear.to_numpy(float)

_test_climate = pd.DataFrame({
    "date":    _dates,
    "x":       -105.5,          # lon — update to match your study area
    "y":        40.0,           # lat
    "ppt_mm":  rng.exponential(3.0, _n),
    "tmean_C": 8 * np.sin(2 * np.pi * (_doy - 80) / 365) + 5 + rng.normal(0, 2, _n),
    "GCM":     "sanity_check",
})

# Use hard-coded params so the test doesn't depend on the rasters being loaded
_test_params = {"Elev": 2400, "Slope": 10, "Aspect": 180,
                "SWC_Max": 150, "J_Temp": 1.5}

_test_result = nps_wbm(
    _test_climate, _test_params,
    pet_method    = PET_METHOD,
    hock_coef     = HOCK_COEF,
    direct_frac   = DIRECT_FRAC,
    return_rate   = RETURN_RATE,
    pet_mult      = PET_MULT,
    soil_mult     = SOIL_MULT,
    shade_coeff   = SHADE_COEFF,
    t_base        = T_BASE,
    to_inches     = False,          # mm for the sanity check
)

_unit = "mm"
print("✓ Sanity check passed — annual water balance totals:")
print(f"  {'Variable':<14}  {'Annual total':>14}")
print(f"  {'-'*30}")
for _col in ["ppt_mm", "RAIN", "SNOW", "MELT", "AET", "RUNOFF", "D"]:
    print(f"  {_col:<14}  {_test_result[_col].sum():>12.1f} {_unit}")

print(f"\n  Peak snowpack : {_test_result['PACK'].max():.1f} {_unit}")
print(f"  Max soil SWC  : {_test_result['SOIL'].max():.1f} {_unit}")
print(f"\n✓ Setup complete — ready to run NPS WBM.\n")

In [ ]:
# NPS WBM Quick-reference: key function signatures
#
# ```python
# # ── Extract site params at your points from the loaded rasters ────────────
# point_params_df = extract_point_params(points_df)
# # points_df needs columns: x (lon), y (lat)
# # Returns:  Elev, Slope, Aspect, SWC_Max, J_Temp  added to points_df
#
# # ── Run for a single point ────────────────────────────────────────────────
# result = nps_wbm(
#     daily_df     = climate_df,        # date, x, y, ppt_mm, tmean_C [, GCM]
#     point_params = point_params_df.iloc[0].to_dict(),
#     pet_method   = PET_METHOD,
#     **{k: v for k, v in globals().items()
#        if k in ("hock_coef","direct_frac","return_rate","pet_mult",
#                 "soil_mult","shade_coeff","t_base","to_inches")},
# )
#
# # ── Run for multiple points / GCMs ───────────────────────────────────────
# results = run_nps_wbm_points(
#     climate_data    = climate_df,
#     point_params_df = point_params_df,
#     pet_method      = PET_METHOD,
#     aggregate       = True,           # False → keep per-point rows
#     ret_final_cond  = False,          # True → chain into next time period
# )
#
# # ── One-call pipeline (load → extract → run) ─────────────────────────────
# results = run_pipeline(
#     climate_data = climate_df,
#     points_df    = points_df,
#     raster_dir   = RASTER_DIR,
#     pet_method   = PET_METHOD,
# )
# ```

**Note:** this notebook reloads notebook 04's calibration outputs (`OBJECTIVE`, `eco_metrics`, and the `metrics_*_vs_*` tables) from `Data/gridmet_cache/`, cached there by a cell at the end of notebook 04. Run notebook 04 at least once (any kernel) before this one -- they no longer need to share a live kernel session.

In [ ]:
# ── Load prerequisites from notebooks 01-04's cache (fresh kernel is fine) ──
flux_towers = pd.read_csv("../Data/geo_data/flux_towers.csv")
climate_gee = pd.read_csv("../Data/gridmet_cache/climate_gee_flux_towers_2016_2023.csv",
                          parse_dates=["date"])
wbm_results = pd.read_csv("../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
                          parse_dates=["date"])
wbm_monthly = pd.read_csv("../Data/gridmet_cache/wbm_monthly_flux_towers_2016_2023.csv",
                          parse_dates=["date_monthly"])
point_params_df = extract_point_params(flux_towers)
sites = sorted(flux_towers["site"].unique())

# ── Load notebook 04's calibration outputs (cached at the end of nb 04) ──────
with open("../Data/gridmet_cache/last_objective.txt") as f:
    OBJECTIVE = f.read().strip()

_metrics_names = [
    "metrics_wbm_vs_openet", "metrics_opt_vs_openet",
    "metrics_wbm_vs_flux", "metrics_opt_vs_flux",
    "metrics_openet_vs_flux", "metrics_cal_vs_openet",
    "metrics_cal_vs_flux", "eco_metrics",
]
_loaded = {}
for _name in _metrics_names:
    _path = f"../Data/gridmet_cache/{_name}_{OBJECTIVE}.csv"
    # keep_default_na=False so a blank 'note' column comes back as "" (as it
    # was in-memory in notebook 04), not NaN -- pandas treats empty CSV
    # fields as NaN by default, which would break `df['note'] == ""` filters.
    _df = pd.read_csv(_path, keep_default_na=False, na_values=[""])
    if "note" in _df.columns:
        _df["note"] = _df["note"].fillna("")
    _loaded[_name] = _df

metrics_wbm_vs_openet  = _loaded["metrics_wbm_vs_openet"]
metrics_opt_vs_openet  = _loaded["metrics_opt_vs_openet"]
metrics_wbm_vs_flux    = _loaded["metrics_wbm_vs_flux"]
metrics_opt_vs_flux    = _loaded["metrics_opt_vs_flux"]
metrics_openet_vs_flux = _loaded["metrics_openet_vs_flux"]
metrics_cal_vs_openet  = _loaded["metrics_cal_vs_openet"]
metrics_cal_vs_flux    = _loaded["metrics_cal_vs_flux"]
eco_metrics            = _loaded["eco_metrics"]


def print_metrics_with_summary(metrics_df, label, metric_cols=None, valid_sites=None):
    """
    Print a per-site metrics table followed by a summary block (mean, median,
    min, max). Copy of the helper defined in notebook 04 -- duplicated here so
    this notebook doesn't depend on notebook 04's in-memory state, only its
    cached CSV outputs (loaded above).
    """
    if metric_cols is None:
        metric_cols = ["slope", "MBE", "MAE", "RMSE", "R2"]

    display_cols = ["site", "n"] + metric_cols + ["note"]
    print(label)
    print(metrics_df[display_cols].to_string(index=False))

    valid = metrics_df[metrics_df["note"] == ""]
    if valid_sites is not None:
        valid = valid[valid["site"].isin(valid_sites)]

    n_valid    = len(valid)
    n_excluded = len(metrics_df) - n_valid
    site_note  = f"n={n_valid} sites"
    if valid_sites is not None:
        site_note += ", restricted to common valid sites"

    if n_valid == 0:
        print("  (no sites with sufficient data for summary)\n")
        return

    summary_rows = []
    for stat_label, fn in [("mean",   lambda x: x.mean()),
                            ("median", lambda x: x.median()),
                            ("min",    lambda x: x.min()),
                            ("max",    lambda x: x.max())]:
        row = {"site": f"── {stat_label} ({site_note})", "n": "", "note": ""}
        for col in metric_cols:
            row[col] = round(fn(valid[col]), 3)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)[display_cols]
    sep = "─" * len(metrics_df[display_cols].to_string(index=False).splitlines()[0])
    print(sep)
    print(summary_df.to_string(index=False, header=False))
    if n_excluded > 0:
        excl_sites = metrics_df[~metrics_df["site"].isin(valid["site"])]["site"].tolist()
        print(f"  (excluded from summary: {excl_sites})")
    print()

print("Loaded base prerequisites (01-03) and calibration outputs "
      f"(04, OBJECTIVE='{OBJECTIVE}') from cache.")


### OLS-scatter facets by ecosystem

The following scatter plots show monthly WBM vs OpenET paired values for each ecosystem type, with OLS regression lines, helping identify whether bias patterns are systematic within particular land-cover classes.

### OLS-scatter facets by ecosystem

The following scatter plots show monthly WBM vs OpenET paired values for each ecosystem type, with OLS regression lines, helping identify whether bias patterns are systematic within particular land-cover classes.

In [ ]:

# ── Comparison pairs: (label, pre-opt col, opt col, color) ───────────────────
COMPARISON_PAIRS = [
    ("WBM vs OpenET",    "RMSE_wbm_default", "RMSE_wbm_opt",      "steelblue"),
    ("WBM vs Flux tower","RMSE_wbm_flux_default","RMSE_wbm_opt_flux","tomato"),
    ("OpenET vs Flux",   None,               "RMSE_openet_flux",   "seagreen"),
]

# Add pre-optimised WBM vs flux (needs computing if not already in eco_metrics)
if "RMSE_wbm_flux_default" not in eco_metrics.columns:
    eco_metrics = eco_metrics.merge(
        metrics_wbm_vs_flux[metrics_wbm_vs_flux["note"] == ""]
        [["site", "RMSE"]]
        .rename(columns={"RMSE": "RMSE_wbm_flux_default"}),
        on="site", how="left"
    )

ecosystems = sorted(eco_metrics["ecosystem"].dropna().unique())
n_eco      = len(ecosystems)

# ── Figure layout: one row per comparison pair ────────────────────────────────
n_pairs = len(COMPARISON_PAIRS)
fig, axes = plt.subplots(
    n_pairs, 2,
    figsize=(7, n_pairs * 2),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.2, 1]},
)

for row_i, (pair_label, pre_col, opt_col, color) in enumerate(COMPARISON_PAIRS):
    ax_strip = axes[row_i, 0]
    ax_bar   = axes[row_i, 1]

    # ── Strip chart ───────────────────────────────────────────────────────────
    jitter = 0.1

    for x_pos, col, style, sublabel in [
        (0, pre_col, "o", "Pre-optimised") if pre_col else (None,)*4,
        (1, opt_col, "D", f"{OBJECTIVE}-optimised"),
    ]:
        if x_pos is None or col is None:
            continue
        vals = eco_metrics[col].dropna().values
        ax_strip.scatter(
            np.full(len(vals), x_pos)
            + np.random.uniform(-jitter, jitter, len(vals)),
            vals,
            color=color, alpha=0.65, s=55,
            marker=style, zorder=4, label=sublabel,
        )
        # Median bar
        med = np.nanmedian(vals)
        ax_strip.hlines(med, x_pos - 0.22, x_pos + 0.22,
                        color=color, linewidth=2.5, zorder=5)
        ax_strip.text(x_pos, med + 1.2, f"{med:.1f}",
                      ha="center", fontsize=8,
                      color=color, fontweight="bold")

    # Connector lines between paired sites
    if pre_col is not None:
        paired = eco_metrics[["site", pre_col, opt_col]].dropna()
        for _, r in paired.iterrows():
            ax_strip.plot(
                [0, 1], [r[pre_col], r[opt_col]],
                color=color, linewidth=0.6, alpha=0.3, zorder=3,
            )

    x_labels = (["Pre-opt", f"{OBJECTIVE}\nopt"]
                 if pre_col else ["—", f"{OBJECTIVE}\nopt"])
    ax_strip.set_xticks([0, 1])
    ax_strip.set_xticklabels(x_labels, fontsize=9)
    ax_strip.set_xlim(-0.5, 1.5)
    ax_strip.set_ylabel("RMSE (mm / month)", fontsize=8)
    ax_strip.set_title(f"{pair_label} — site-level RMSE\n"
                        "(bar = median, lines connect same site)",
                        fontsize=9, fontweight="bold")
    ax_strip.spines[["top", "right"]].set_visible(False)
    ax_strip.set_ylim(bottom=0)
    ax_strip.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax_strip.legend(fontsize=8, frameon=False, loc="upper right")

    # ── Grouped bar chart by ecosystem ────────────────────────────────────────
    x_eco = np.arange(n_eco)
    bar_w = 0.35

    for offset, col, style, sublabel in [
        (-bar_w / 2, pre_col, "///", "Pre-optimised"),
        ( bar_w / 2, opt_col, "",   f"{OBJECTIVE}-optimised"),
    ]:
        if col is None:
            continue
        eco_means = [
            eco_metrics[eco_metrics["ecosystem"] == eco][col].mean()
            for eco in ecosystems
        ]
        bars = ax_bar.bar(
            x_eco + offset, eco_means,
            width=bar_w, color=color,
            alpha=0.55 if style else 0.85,
            hatch=style,
            label=sublabel,
            edgecolor="white" if not style else color,
            linewidth=0.5,
        )
        for bar, val in zip(bars, eco_means):
            if not np.isnan(val):
                ax_bar.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.8,
                    f"{val:.0f}",
                    ha="center", va="bottom",
                    fontsize=6.5, color="black",
                )

    # n= labels
    for i, eco in enumerate(ecosystems):
        n = eco_metrics[eco_metrics["ecosystem"] == eco]["site"].nunique()
        ax_bar.text(i, -5, f"n={n}",
                    ha="center", fontsize=7, color="grey")

    ax_bar.set_xticks(x_eco)
    ax_bar.set_xticklabels(
        [e.replace(" ", "\n").replace("/", "/\n") for e in ecosystems],
        fontsize=8,
    )
    ax_bar.set_ylabel("Mean RMSE (mm / month)", fontsize=8)
    ax_bar.set_title(f"{pair_label} — mean RMSE by ecosystem",
                      fontsize=9, fontweight="bold")
    ax_bar.spines[["top", "right"]].set_visible(False)
    ax_bar.set_ylim(bottom=0)
    ax_bar.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax_bar.legend(fontsize=8, frameon=False, loc="upper right")

fig.suptitle(
    f"RMSE before vs after {OBJECTIVE} calibration",
    fontsize=12, fontweight="bold",
)

save_path = f"../Data/open_et/rmse_ecosystem_prepost_{OBJECTIVE}.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n── RMSE before vs after by ecosystem ───────────────────────────────────")
for pair_label, pre_col, opt_col, _ in COMPARISON_PAIRS:
    print(f"\n  {pair_label}:")
    rows = []
    for eco in ecosystems:
        sub  = eco_metrics[eco_metrics["ecosystem"] == eco]
        pre  = sub[pre_col].mean() if pre_col else np.nan
        opt  = sub[opt_col].mean()
        delta = opt - pre if not np.isnan(pre) else np.nan
        rows.append({
            "ecosystem": eco,
            "n":         sub["site"].nunique(),
            "RMSE_pre":  round(pre, 1),
            "RMSE_opt":  round(opt, 1),
            "Δ":         round(delta, 1) if not np.isnan(delta) else "—",
        })
    print(pd.DataFrame(rows).to_string(index=False))

### Before vs. after optimisation — fair site-matched comparison

To fairly compare default and PET-optimised performance, the table below restricts the summary to sites that have valid metrics under *both* configurations. This prevents the aggregate statistics from being skewed by sites that only appear in one comparison.

In [ ]:
# ── Fair comparison: only sites with valid metrics in BOTH default and optimised ──
valid_default = set(metrics_wbm_vs_openet[metrics_wbm_vs_openet["note"] == ""]["site"])
valid_opt     = set(metrics_opt_vs_openet[metrics_opt_vs_openet["note"] == ""]["site"])
common_sites  = valid_default & valid_opt

print(f"Sites in default only : {valid_default - common_sites}")
print(f"Sites in optimised only: {valid_opt - common_sites}")
print(f"Common sites ({len(common_sites)}): {sorted(common_sites)}\n")

for label, mdf in [("Default WBM", metrics_wbm_vs_openet),
                   (f"Optimised ({OBJECTIVE})", metrics_opt_vs_openet)]:
    sub = mdf[mdf["site"].isin(common_sites)]
    print(f"  {label}:")
    for m in ["slope", "MBE", "MAE", "RMSE", "R2"]:
        print(f"    {m:<5} = {sub[m].mean():>8.3f}")

Now, look at model performance compared to elevation.

In [ ]:
# ── Elevation vs. Model Performance ──────────────────────────────────────────

# ── Build per-site lookup: elevation + ecosystem ──────────────────────────────
site_meta = (
    point_params_df[["site", "Elev"]]
    .merge(flux_towers[["site", "ecosystem"]], on="site", how="left")
)

# ── Merge elevation/ecosystem into each metrics dataframe ─────────────────────
def add_meta(metrics_df):
    return (
        metrics_df[metrics_df["note"] == ""]
        .merge(site_meta, on="site", how="left")
        .dropna(subset=["Elev", "RMSE", "MAE", "MBE"])
    )

m_wbm_openet  = add_meta(metrics_wbm_vs_openet).assign(comparison="WBM vs OpenET")
m_openet_flux = add_meta(metrics_openet_vs_flux).assign(comparison="OpenET vs Flux")
m_wbm_flux    = add_meta(metrics_wbm_vs_flux).assign(comparison="WBM vs Flux")
m_opt_openet  = add_meta(metrics_opt_vs_openet).assign(comparison=f"NSE Opt. WBM\nvs OpenET")

all_meta    = [m_openet_flux, m_wbm_flux, m_wbm_openet, m_opt_openet]
comparisons = ["OpenET vs Flux", "WBM vs Flux", "WBM vs OpenET",  f"NSE Opt. WBM\nvs OpenET"]
metrics_to_plot = ["RMSE", "MAE", "MBE"]

# ── Ecosystem color palette ───────────────────────────────────────────────────
ecosystems = site_meta["ecosystem"].dropna().unique()
palette    = plt.cm.tab10.colors
eco_colors = {eco: palette[i % len(palette)] for i, eco in enumerate(sorted(ecosystems))}

# ── Figure: 3 rows (metric) × 4 cols (comparison) ────────────────────────────
fig, axes = plt.subplots(
    len(metrics_to_plot), len(comparisons),
    figsize=(7, 5),          # wider to accommodate 4th column
    constrained_layout=True,
    sharey="row",
)

for col_i, (df, comp) in enumerate(zip(all_meta, comparisons)):
    for row_i, metric in enumerate(metrics_to_plot):
        ax = axes[row_i, col_i]

        for _, row in df.iterrows():
            eco   = row["ecosystem"] if pd.notna(row["ecosystem"]) else "Unknown"
            color = eco_colors.get(eco, "grey")
            ax.scatter(
                row["Elev"], row[metric],
                color      = color,
                s          = 20,
                edgecolors = "white",
                linewidths = 0.5,
                zorder     = 4,
            )

        # Reference line at 0 for MBE
        if metric == "MBE":
            ax.axhline(0, color="grey", linewidth=0.7, linestyle="--", zorder=2)

        # Light trend line
        if len(df) >= 3:
            z   = np.polyfit(df["Elev"], df[metric], 1)
            p   = np.poly1d(z)
            xr  = np.linspace(df["Elev"].min(), df["Elev"].max(), 100)
            ax.plot(xr, p(xr), color="grey", linewidth=0.8,
                    linestyle="--", alpha=0.6, zorder=3)

        ax.set_xlabel("Elevation (m)", fontsize=8)
        ax.set_ylabel(f"{metric} (mm/month)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)

        if row_i == 0:
            ax.set_title(comp, fontsize=9, fontweight="bold", pad=6)

# ── Shared ecosystem legend at figure bottom ──────────────────────────────────
legend_handles = [
    mlines.Line2D([], [], marker="o", color=eco_colors[eco], linestyle="none",
                  markersize=6, markeredgecolor="white", markeredgewidth=0.5,
                  label=eco)
    for eco in sorted(eco_colors)
]

fig.legend(
    handles        = legend_handles,
    loc            = "lower center",
    ncol           = min(len(legend_handles), 5),
    fontsize       = 7,
    frameon        = True,
    framealpha     = 0,
    edgecolor      = "grey",
    title          = "Ecosystem",
    title_fontsize = 8,
    bbox_to_anchor = (0.5, 0),
)
fig.get_layout_engine().set(rect=(0, 0.07, 1, 1))

plt.savefig("../Data/open_et/elevation_vs_performance.png", dpi=600, bbox_inches="tight")
plt.show()

Now, we'll evaluate how the WBM AET error relates to the overall water balance by comparing AET error to total preciptiation.

In [ ]:
# ── AET Error as a Fraction of GridMET Precipitation (Water Balance Context) ──
#
# GridMET precipitation (ppt_mm) is always in mm in wbm_monthly regardless of
# the TO_INCHES flag (it is a WBM input, not an output).
#
# Error metrics (MAE, RMSE, MBE) from compute_metrics() are in mm month⁻¹.
# To compare against the water balance, both are annualised:
#   annual_error_equiv (mm yr⁻¹) = metric × 12
#   mean_annual_P      (mm yr⁻¹) = site total P / n years
# Relative error = annual_error_equiv / mean_annual_P  (dimensionless fraction)


# ── 1. Mean annual GridMET precipitation per site ─────────────────────────────
annual_ppt = (
    wbm_monthly
    .groupby(["site", "year"])["ppt_mm"]
    .sum()                      # total P for each year at each site
    .reset_index()
    .groupby("site")["ppt_mm"]
    .mean()                     # mean annual P across all years on record
    .rename("mean_annual_P_mm")
    .reset_index()
)

print("Mean annual GridMET precipitation by site (mm yr⁻¹):")
print(annual_ppt.sort_values("mean_annual_P_mm").to_string(index=False))

# ── 2. Build a unified relative-error table across all comparisons ─────────────
comparison_map = {
    "WBM vs OpenET":         metrics_wbm_vs_openet,
    "WBM vs Flux":           metrics_wbm_vs_flux,
    "OpenET vs Flux":        metrics_openet_vs_flux,
    f"NSE Opt. WBM\nvs OpenET": metrics_opt_vs_openet,
    f"NSE Opt. WBM\nvs Flux":   metrics_opt_vs_flux,
}

rel_frames = []
for label, mdf in comparison_map.items():
    df = (
        mdf[mdf["note"] == ""]          # drop sites with insufficient data
        .merge(annual_ppt, on="site", how="left")
        .merge(
            site_meta[["site", "Elev", "ecosystem"]],
            on="site", how="left"
        )
    )
    # Annualise monthly metrics (mm month⁻¹ → mm yr⁻¹ equivalent)
    # then express as a fraction of mean annual P
    df["rel_MAE"]  = (df["MAE"]  * 12) / df["mean_annual_P_mm"]
    df["rel_RMSE"] = (df["RMSE"] * 12) / df["mean_annual_P_mm"]
    df["rel_MBE"]  = (df["MBE"]  * 12) / df["mean_annual_P_mm"]
    df["comparison"] = label
    rel_frames.append(df)

rel_all = pd.concat(rel_frames, ignore_index=True)

# ── 3. Summary table ─────────────────────────────────────────────────────────
print("\n── Relative error metrics (fraction of mean annual GridMET P) ───────────")
summary = (
    rel_all
    .groupby("comparison")[["rel_MAE", "rel_RMSE", "rel_MBE"]]
    .agg(["mean", "min", "max"])
    .round(3)
)
summary.columns = ["_".join(c) for c in summary.columns]
print(summary.to_string())

# ── 4. Figure: relative MAE and RMSE per site, faceted by comparison ──────────
comparisons = list(comparison_map.keys())
n_comp      = len(comparisons)

# Ecosystem colour palette (consistent with earlier figures)
ecosystems  = sorted(rel_all["ecosystem"].dropna().unique())
palette     = plt.cm.tab10.colors
eco_colors  = {eco: palette[i % len(palette)] for i, eco in enumerate(ecosystems)}

fig, axes = plt.subplots(
    2, n_comp,
    figsize=(7, 4),
    constrained_layout=True,
    sharey="row",
)

metric_rows  = [("rel_MAE",  "MAE / P (fraction)"),
                ("rel_RMSE", "RMSE / P (fraction)")]

for col_i, comp in enumerate(comparisons):
    sub = rel_all[rel_all["comparison"] == comp].copy()

    for row_i, (metric, ylabel) in enumerate(metric_rows):
        ax = axes[row_i, col_i]

        # Sort sites by mean annual P for a consistent x-axis order
        sub_sorted = sub.sort_values("mean_annual_P_mm").dropna(
            subset=[metric, "mean_annual_P_mm"]
        )

        for _, row in sub_sorted.iterrows():
            eco   = row["ecosystem"] if pd.notna(row["ecosystem"]) else "Unknown"
            color = eco_colors.get(eco, "grey")
            ax.scatter(
                row["mean_annual_P_mm"],
                row[metric],
                color=color, s=22,
                edgecolors="white", linewidths=0.5,
                zorder=4,
            )

        # Light trend line if enough points
        valid = sub_sorted.dropna(subset=[metric, "mean_annual_P_mm"])
        if len(valid) >= 3:
            z  = np.polyfit(valid["mean_annual_P_mm"], valid[metric], 1)
            p  = np.poly1d(z)
            xr = np.linspace(valid["mean_annual_P_mm"].min(),
                             valid["mean_annual_P_mm"].max(), 100)
            ax.plot(xr, p(xr), color="grey",
                    linewidth=0.8, linestyle="--", alpha=0.6, zorder=3)

        ax.set_xlabel("Mean annual P (mm yr⁻¹)", fontsize=7)
        ax.set_ylabel(ylabel, fontsize=7)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_ylim(bottom=0)

        if row_i == 0:
            ax.set_title(comp, fontsize=8, fontweight="bold", pad=5)

# ── Shared ecosystem legend ───────────────────────────────────────────────────
legend_handles = [
    mlines.Line2D([], [], marker="o", color=eco_colors[eco],
                  linestyle="none", markersize=6,
                  markeredgecolor="white", markeredgewidth=0.5,
                  label=eco)
    for eco in sorted(eco_colors)
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=min(len(legend_handles), 5),
    fontsize=7,
    frameon=True,
    framealpha=0,
    edgecolor="grey",
    title="Ecosystem",
    title_fontsize=8,
    bbox_to_anchor=(0.5, 0),
)
fig.get_layout_engine().set(rect=(0, 0.09, 1, 0.9))
fig.suptitle(
    "AET Error as Fraction of Mean Annual Precipitation",
    fontsize=10, fontweight="bold",
)

plt.savefig("../Data/open_et/aet_error_fraction_precip.png", dpi=600, bbox_inches="tight")
plt.show()

# ── 5. Per-site table: absolute vs relative metrics for WBM comparisons ────────
print("\n── Per-site error fraction of annual precipitation ──────────────────────")
for comp in ["WBM vs OpenET", "WBM vs Flux"]:
    sub = rel_all[rel_all["comparison"] == comp].sort_values("mean_annual_P_mm")
    if sub.empty:
        continue
    print(f"\n  {comp}:")
    print(
        sub[["site", "ecosystem", "mean_annual_P_mm",
             "MAE", "rel_MAE", "RMSE", "rel_RMSE", "MBE", "rel_MBE"]]
        .rename(columns={
            "mean_annual_P_mm": "P (mm/yr)",
            "MAE":  "MAE (mm/mo)",  "rel_MAE":  "MAE/P",
            "RMSE": "RMSE (mm/mo)", "rel_RMSE": "RMSE/P",
            "MBE":  "MBE (mm/mo)",  "rel_MBE":  "MBE/P",
        })
        .round({"P (mm/yr)": 0, "MAE (mm/mo)": 1, "MAE/P": 3,
                "RMSE (mm/mo)": 1, "RMSE/P": 3,
                "MBE (mm/mo)": 1,  "MBE/P": 3})
        .to_string(index=False)
    )

## Final statistics summary and CSV export

Now that all bias-correction approaches have been applied — including the simple OLS post-hoc multiplier (cell above) and the within-model PET multiplier optimisation — we compile a single comprehensive metrics table covering every comparison variant and save it to disk.

The six comparisons saved are:
1. **WBM vs OpenET** — baseline default model
2. **OpenET vs Flux** — OpenET accuracy reference
3. **WBM vs Flux** — baseline model vs ground truth
4. **OLS-Calibrated WBM vs OpenET** — post-hoc scaling correction
5. **OLS-Calibrated WBM vs Flux** — post-hoc correction vs ground truth
6. **PET-Optimised WBM vs OpenET** — within-model PET multiplier calibration
7. **PET-Optimised WBM vs Flux** — within-model calibration vs ground truth

In [ ]:
# ── Compile final comprehensive metrics across all correction approaches ───────
#
# This cell must be run AFTER:
#   • Cell 52  – OLS post-hoc calibration  (metrics_cal_vs_openet / _flux)
#   • Cell 57  – PET optimisation rebuild  (metrics_opt_vs_openet / _flux)
#
# All per-site metrics DataFrames are tagged with a "comparison" label before
# concatenation so each row is unambiguously identified in the output CSV.

# ── Tag OLS-calibrated metrics if "comparison" column is absent ───────────────
for _mdf, _label in [
    (metrics_cal_vs_openet, "OLS-Calibrated WBM vs OpenET"),
    (metrics_cal_vs_flux,   "OLS-Calibrated WBM vs Flux"),
]:
    if "comparison" not in _mdf.columns:
        _mdf.insert(0, "comparison", _label)
    else:
        _mdf["comparison"] = _label

# ── Tag PET-optimised metrics (comparison label includes objective name) ──────
for _mdf, _label in [
    (metrics_opt_vs_openet, f"PET-Optimised ({OBJECTIVE}) vs OpenET"),
    (metrics_opt_vs_flux,   f"PET-Optimised ({OBJECTIVE}) vs Flux"),
]:
    if "comparison" not in _mdf.columns:
        _mdf.insert(0, "comparison", _label)
    else:
        _mdf["comparison"] = _label

# ── Ensure baseline metrics also carry their labels ───────────────────────────
metrics_wbm_vs_openet["comparison"]  = "WBM vs OpenET"
metrics_openet_vs_flux["comparison"] = "OpenET vs Flux"
metrics_wbm_vs_flux["comparison"]    = "WBM vs Flux"

# ── Combine all comparisons ───────────────────────────────────────────────────
metrics_all_final = pd.concat(
    [
        metrics_wbm_vs_openet,
        metrics_openet_vs_flux,
        metrics_wbm_vs_flux,
        metrics_cal_vs_openet,
        metrics_cal_vs_flux,
        metrics_opt_vs_openet,
        metrics_opt_vs_flux,
    ],
    ignore_index=True,
)

# ── Common valid site sets for fair aggregate comparisons ─────────────────────
# Use intersection of valid sites for each comparison pair so aggregate stats
# are computed over the same sites across correction methods.
_valid = lambda mdf: set(mdf[mdf["note"] == ""]["site"])

common_vs_openet_final = (
    _valid(metrics_wbm_vs_openet)
    & _valid(metrics_cal_vs_openet)
    & _valid(metrics_opt_vs_openet)
)
common_vs_flux_final = (
    _valid(metrics_wbm_vs_flux)
    & _valid(metrics_cal_vs_flux)
    & _valid(metrics_opt_vs_flux)
)
common_openet_flux_final = _valid(metrics_openet_vs_flux)

comparison_site_map_final = {
    "WBM vs OpenET":                          common_vs_openet_final,
    "OpenET vs Flux":                         common_openet_flux_final,
    "WBM vs Flux":                            common_vs_flux_final,
    "OLS-Calibrated WBM vs OpenET":           common_vs_openet_final,
    "OLS-Calibrated WBM vs Flux":             common_vs_flux_final,
    f"PET-Optimised ({OBJECTIVE}) vs OpenET": common_vs_openet_final,
    f"PET-Optimised ({OBJECTIVE}) vs Flux":   common_vs_flux_final,
}

# ── Print final per-site tables ───────────────────────────────────────────────
print("\n══ FINAL COMPREHENSIVE METRICS — ALL BIAS-CORRECTION APPROACHES ══════")
for _label, _mdf, _common in [
    ("WBM vs OpenET",                 metrics_wbm_vs_openet,  common_vs_openet_final),
    ("OpenET vs Flux",                metrics_openet_vs_flux, common_openet_flux_final),
    ("WBM vs Flux",                   metrics_wbm_vs_flux,    common_vs_flux_final),
    ("OLS-Calibrated WBM vs OpenET",  metrics_cal_vs_openet,  common_vs_openet_final),
    ("OLS-Calibrated WBM vs Flux",    metrics_cal_vs_flux,    common_vs_flux_final),
    (f"PET-Optimised ({OBJECTIVE}) vs OpenET", metrics_opt_vs_openet, common_vs_openet_final),
    (f"PET-Optimised ({OBJECTIVE}) vs Flux",   metrics_opt_vs_flux,   common_vs_flux_final),
]:
    print_metrics_with_summary(
        _mdf, f"── {_label} ─────────────────────────────────────────",
        valid_sites=_common,
    )

# ── Cross-site aggregate table (filtered to common site sets) ─────────────────
metrics_all_final_filtered = metrics_all_final[
    metrics_all_final.apply(
        lambda r: r["site"] in comparison_site_map_final.get(r["comparison"], set()),
        axis=1,
    )
]

print("── Cross-site aggregate metrics (all corrections, common valid sites) ────")
agg_final = (
    metrics_all_final_filtered[metrics_all_final_filtered["note"] == ""]
    .groupby("comparison")
    .agg(
        n_sites    = ("site",  "count"),
        slope_mean = ("slope", "mean"),
        MBE_mean   = ("MBE",   "mean"),
        MAE_mean   = ("MAE",   "mean"),
        RMSE_mean  = ("RMSE",  "mean"),
        R2_mean    = ("R2",    "mean"),
    )
    .round(3)
    .reset_index()
)
print(agg_final.to_string(index=False))

# ── Save to CSV ───────────────────────────────────────────────────────────────
metrics_all_final.to_csv("../Data/open_et/et_comparison_metrics.csv", index=False)
agg_final.to_csv("../Data/open_et/et_comparison_metrics_agg.csv", index=False)
print("\nSaved to Data/open_et/et_comparison_metrics.csv")
print("Saved to Data/open_et/et_comparison_metrics_agg.csv")


To further explore the cause for underestimation of AET, we'll also look at PET. GridMET PET (Penman-Monteith) provides a quick comparison mechanism that may help explain whether low PET was the original issue. Given that we're applying a multiplier to PET, this would justifiy the correction approach. 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PET DIAGNOSIS: WBM Oudin PET vs GridMET Reference ET (etr)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Hypothesis: Low WBM AET may be driven by low Oudin PET rather than soil-
# moisture limitation alone.  To test this, we compare:
#
#   • WBM PET (Oudin)   → PET_mod column in wbm_results, daily → summed monthly
#   • GridMET PET (etr) → Penman-Monteith alfalfa reference ET, already in
#                         wbm_monthly as etr_gridmet_mm (mm / month)
#
# Both expressed in mm / month.  A persistent negative bias in WBM PET relative
# to GridMET etr would confirm that Oudin under-estimates the ET demand and
# directly suppresses AET.
# ───────────────────────────────────────────────────────────────────────────────

# ── Step 1: Aggregate WBM PET_mod to monthly (mm) ────────────────────────────
# PET_mod = PET × pet_mult — the demand value that actually caps AET in the
# bucket model.  Fall back to PET if PET_mod is absent.
_pet_col = "PET_mod" if "PET_mod" in wbm_results.columns else "PET"
_mm_scale = 25.4 if TO_INCHES else 1.0   # WBM output unit → mm

pet_wbm_monthly = (
    wbm_results
    .assign(date_monthly=lambda d: pd.to_datetime(
        d["date"].dt.to_period("M").dt.to_timestamp()
    ))
    .groupby(["site", "date_monthly"], as_index=False)[_pet_col]
    .sum()
    .rename(columns={_pet_col: "pet_wbm_mm"})
)
pet_wbm_monthly["pet_wbm_mm"] *= _mm_scale

print(f"WBM PET source column : '{_pet_col}'  (×{_mm_scale} → mm)")
print(f"Monthly rows          : {len(pet_wbm_monthly):,}")

# ── Step 2: Pull GridMET etr from wbm_monthly ────────────────────────────────
# etr_gridmet_mm is already a daily sum → monthly total in mm.
# If it is somehow absent, fall back to climate_gee and re-aggregate.
if "etr_gridmet_mm" in wbm_monthly.columns:
    gm_pet_monthly = wbm_monthly[["site", "date_monthly", "etr_gridmet_mm"]].copy()
    print("GridMET etr source    : wbm_monthly['etr_gridmet_mm']")
else:
    # Fallback: aggregate from raw daily climate data
    gm_pet_monthly = (
        climate_gee
        .assign(date_monthly=lambda d: pd.to_datetime(
            d["date"].dt.to_period("M").dt.to_timestamp()
        ))
        .groupby(["site", "date_monthly"], as_index=False)["etr_gridmet_mm"]
        .sum()
    )
    print("GridMET etr source    : climate_gee (re-aggregated)")

# ── Step 3: Merge ─────────────────────────────────────────────────────────────
pet_compare = (
    pet_wbm_monthly
    .merge(gm_pet_monthly, on=["site", "date_monthly"], how="inner")
    .dropna(subset=["pet_wbm_mm", "etr_gridmet_mm"])
    .sort_values(["site", "date_monthly"])
    .reset_index(drop=True)
)

print(f"\nMerged rows (site-months): {len(pet_compare):,}")
print(pet_compare[["site", "date_monthly", "pet_wbm_mm", "etr_gridmet_mm"]].head(8)
      .to_string(index=False))

# ── Step 4: Per-site summary statistics ───────────────────────────────────────
print("\n══ Monthly PET — WBM Oudin vs GridMET etr ══════════════════════════════")
hdr = f"{'Site':<14}  {'n':>5}  {'WBM mean':>10}  {'GM mean':>9}  "  \
      f"{'MBE':>9}  {'MAE':>9}  {'R²':>6}"
print(hdr)
print("─" * len(hdr))

stats_rows = []
for site in sorted(pet_compare["site"].unique()):
    sub  = pet_compare[pet_compare["site"] == site]
    diff = sub["pet_wbm_mm"] - sub["etr_gridmet_mm"]
    mbe  = diff.mean()
    mae  = diff.abs().mean()
    r2   = sub["pet_wbm_mm"].corr(sub["etr_gridmet_mm"]) ** 2
    print(f"{site:<14}  {len(sub):>5}  {sub['pet_wbm_mm'].mean():>10.1f}  "
          f"{sub['etr_gridmet_mm'].mean():>9.1f}  {mbe:>+9.1f}  {mae:>9.1f}  {r2:>6.3f}")
    stats_rows.append(dict(site=site, n=len(sub),
                           wbm_mean=sub["pet_wbm_mm"].mean(),
                           gm_mean=sub["etr_gridmet_mm"].mean(),
                           MBE=mbe, MAE=mae, R2=r2))

# Overall mean
all_mbe = (pet_compare["pet_wbm_mm"] - pet_compare["etr_gridmet_mm"]).mean()
all_mae = (pet_compare["pet_wbm_mm"] - pet_compare["etr_gridmet_mm"]).abs().mean()
all_r2  = pet_compare["pet_wbm_mm"].corr(pet_compare["etr_gridmet_mm"]) ** 2
print("─" * len(hdr))
print(f"{'ALL SITES':<14}  {len(pet_compare):>5}  {pet_compare['pet_wbm_mm'].mean():>10.1f}  "
      f"{pet_compare['etr_gridmet_mm'].mean():>9.1f}  {all_mbe:>+9.1f}  {all_mae:>9.1f}  {all_r2:>6.3f}")
print("\n  MBE < 0  → Oudin PET is LOWER than GridMET etr  (likely under-estimates demand)")
print("  MBE > 0  → Oudin PET is HIGHER than GridMET etr")

# ── Step 5: Time-series facet plot ────────────────────────────────────────────
sites_list = sorted(pet_compare["site"].unique())
n_sites    = len(sites_list)
n_cols     = min(3, n_sites)
n_rows     = int(np.ceil(n_sites / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 3.5 * n_rows),
                         constrained_layout=True, sharey=False)
axes_flat = np.array(axes).flatten()

for ax, site in zip(axes_flat, sites_list):
    sub = pet_compare[pet_compare["site"] == site].sort_values("date_monthly")
    ax.plot(sub["date_monthly"], sub["etr_gridmet_mm"],
            color="#1a5e2a", linewidth=1.4, label="GridMET etr (PM alfalfa)")
    ax.plot(sub["date_monthly"], sub["pet_wbm_mm"],
            color="#d94f0a", linewidth=1.4, linestyle="--", label="WBM PET (Oudin)")
    ax.fill_between(sub["date_monthly"],
                    sub["pet_wbm_mm"], sub["etr_gridmet_mm"],
                    where=(sub["etr_gridmet_mm"] > sub["pet_wbm_mm"]),
                    color="#1a5e2a", alpha=0.12, label="GM > WBM (gap)")
    ax.fill_between(sub["date_monthly"],
                    sub["pet_wbm_mm"], sub["etr_gridmet_mm"],
                    where=(sub["pet_wbm_mm"] >= sub["etr_gridmet_mm"]),
                    color="#d94f0a", alpha=0.12)

    s = next(r for r in stats_rows if r["site"] == site)
    ax.set_title(f"{site}  |  MBE = {s['MBE']:+.0f} mm/mo  |  R² = {s['R2']:.3f}",
                 fontsize=9, fontweight="bold")
    ax.set_ylabel("PET (mm / month)", fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax.tick_params(labelsize=8)
    ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)

axes_flat[0].legend(fontsize=8, loc="upper right")
for ax in axes_flat[n_sites:]:
    ax.set_visible(False)

fig.suptitle("Monthly PET: WBM Oudin vs GridMET etr (Penman-Monteith alfalfa)\n"
             "Shaded gap = GridMET > WBM (Oudin under-estimates demand)",
             fontsize=12, fontweight="bold")
plt.savefig("../Data/open_et/pet_comparison_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Step 6: Scatter facet plot ────────────────────────────────────────────────
fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows),
                            constrained_layout=True)
axes2_flat = np.array(axes2).flatten()

for ax, site in zip(axes2_flat, sites_list):
    sub    = pet_compare[pet_compare["site"] == site]
    xy_max = max(sub["etr_gridmet_mm"].max(), sub["pet_wbm_mm"].max()) * 1.08

    ax.scatter(sub["etr_gridmet_mm"], sub["pet_wbm_mm"],
               c=sub["date_monthly"].dt.month,   # colour by month for seasonality
               cmap="twilight_shifted", alpha=0.75, s=20, edgecolors="none")
    ax.plot([0, xy_max], [0, xy_max], "k--", linewidth=0.9, label="1:1")
    ax.set_xlim(0, xy_max); ax.set_ylim(0, xy_max)
    ax.set_xlabel("GridMET etr (mm/mo)", fontsize=8)
    ax.set_ylabel("WBM PET Oudin (mm/mo)", fontsize=8)

    s = next(r for r in stats_rows if r["site"] == site)
    ax.set_title(f"{site}\nMBE={s['MBE']:+.0f} mm  MAE={s['MAE']:.0f} mm  R²={s['R2']:.3f}",
                 fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

axes2_flat[0].legend(fontsize=8)
# Shared colorbar for month
sm = plt.cm.ScalarMappable(cmap="twilight_shifted",
                            norm=plt.Normalize(vmin=1, vmax=12))
sm.set_array([])
cbar = fig2.colorbar(sm, ax=axes2_flat[:n_sites], shrink=0.6, pad=0.02)
cbar.set_label("Month", fontsize=9)
cbar.set_ticks([1, 3, 6, 9, 12])
cbar.set_ticklabels(["Jan", "Mar", "Jun", "Sep", "Dec"])

for ax in axes2_flat[n_sites:]:
    ax.set_visible(False)

fig2.suptitle("WBM Oudin PET vs GridMET etr — monthly scatter\n"
              "(points below 1:1 line → Oudin under-estimates relative to GridMET)",
              fontsize=12, fontweight="bold")
plt.savefig("../Data/open_et/pet_comparison_scatter.png", dpi=150, bbox_inches="tight")
plt.show()